# 머신러닝 알고리즘의 확률론적 근거

이 강의노트는 **42H: 머신러닝** 1장~4장에서 다룬 알고리즘들의 확률론적 배경을 설명합니다.

- **RMSE**: 왜 오차를 제곱해서 평균 내는가? (1장, 2장)
- **시그모이드**: 왜 출력값이 확률인가? (4장 로지스틱 회귀)
- **소프트맥스**: 왜 출력값들의 합이 1인가? (4장 소프트맥스 회귀)

> 수식보다 직관을 중심으로 설명하며, 필요한 경우 최소한의 수식을 함께 제시합니다.

---
## 1. RMSE와 정규분포

### 1.1 오차란 무엇인가?

2장에서 캘리포니아 주택 가격을 예측하는 회귀 모델을 훈련시켰습니다.  
모델이 어떤 구역의 중위 주택 가격을 **3억**으로 예측했는데 실제 가격이 **3억 2천만원**이라면, 오차는 **2천만원**입니다.

모든 샘플에 대해 이 오차를 계산할 수 있고, 이 오차들의 분포가 어떤 모양인지가 중요합니다.

### 1.2 오차는 왜 정규분포를 따르는가?

모델이 예측을 틀리는 이유는 다양합니다.

- 집 근처에 새로 생긴 공원
- 집주인의 급매 여부
- 그날의 부동산 시장 분위기
- 햇빛이 잘 드는 방향
- 수십 수백 가지의 작은 요인들...

이 요인들은 **서로 독립적**이고, 각각 오차를 조금씩 위아래로 밀어냅니다.  
최종 오차는 이 작은 요인들이 모두 합쳐진 결과입니다.

**중심극한정리**에 의하면, 서로 독립적인 확률변수들의 합은 개수가 충분히 많을 때 정규분포에 가까워집니다.  
따라서 오차가 정규분포를 따른다고 가정하는 것이 자연스럽습니다.

정규분포를 따르는 오차는 다음과 같은 특징을 가집니다.

- 오차가 **0 근처**에 가장 많이 몰려 있다 → 모델이 대체로 맞게 예측한다
- 오차가 **클수록 점점 드물어진다** → 크게 틀리는 경우는 가끔만 있다
- **양수 오차와 음수 오차가 대칭적**으로 나타난다 → 높게 틀리는 것과 낮게 틀리는 것이 비슷하게 나타난다

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# 독립적인 작은 요인 100개를 합쳐서 오차를 시뮬레이션
np.random.seed(42)
n_samples = 10000
n_factors = 100

# 각 요인은 균등분포를 따름 (정규분포가 아님)
factors = np.random.uniform(-1, 1, size=(n_samples, n_factors))

# 오차 = 요인들의 합
errors = factors.sum(axis=1)

x = np.linspace(errors.min(), errors.max(), 200)
pdf = stats.norm.pdf(x, errors.mean(), errors.std())

plt.figure(figsize=(8, 4))
plt.hist(errors, bins=60, density=True, alpha=0.6, color='steelblue', label='Simulated Error Distribution')
plt.plot(x, pdf, 'r-', linewidth=2, label='Normal Distribution Curve')
plt.title('Sum of 100 Independent Factors → Approaches Normal Distribution (Central Limit Theorem)')
plt.xlabel('Error')
plt.ylabel('Density')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
print('Each factor follows a uniform distribution, but the sum approximates a normal distribution.')

### 1.3 정규분포의 핵심 성질

정규분포의 확률밀도함수는 다음과 같습니다.

$$f(e) = \frac{1}{\sigma\sqrt{2\pi}} \exp\left(-\frac{e^2}{2\sigma^2}\right)$$

핵심은 뒷부분입니다.

$$\exp\left(-\frac{e^2}{2\sigma^2}\right)$$

오차 $e$가 커질수록 이 값이 **폭발적으로 작아집니다**.
단순히 비례해서 줄어드는 게 아니라, $e^2$이 지수 위에 올라가 있기 때문입니다.

In [ ]:
import numpy as np

# 오차 크기에 따른 확률밀도 변화
errors = np.array([0, 1, 2, 3, 4, 5])
sigma = 1.0
prob_density = np.exp(-errors**2 / (2 * sigma**2))

print('Probability Density by Error Size (Normalized)')
print('-' * 40)
for e, p in zip(errors, prob_density):
    bar = '█' * int(p * 30)
    print(f'Error = {e:2d}  →  {p:.6f}  {bar}')

print()
print('Even a small increase in error causes the probability density to drop drastically.')
print('This is why RMSE is highly sensitive to large errors.')

### 1.4 정규분포 가정으로부터 RMSE가 유도되는 과정

오차가 정규분포를 따른다고 가정하면, 모델의 목표를 이렇게 말할 수 있습니다.

> **"주어진 데이터가 나올 가능성을 최대로 만드는 파라미터를 찾아라."**  
> (이를 **최대우도추정, MLE** 라고 합니다)

모든 샘플의 오차가 독립적이라면, 전체 데이터가 나올 확률은 각 샘플의 확률을 곱한 것입니다.

$$P \propto \exp(-e_1^2) \times \exp(-e_2^2) \times \cdots \times \exp(-e_n^2) = \exp\left(-(e_1^2 + e_2^2 + \cdots + e_n^2)\right)$$

이 값을 **최대화**하려면, 지수 안의 값을 **최소화**하면 됩니다.

$$e_1^2 + e_2^2 + \cdots + e_n^2 \text{ 를 최소화}$$

이것이 바로 **오차의 제곱합을 최소화**하는 것, 즉 **MSE를 최소화**하는 것과 같습니다.  
여기에 루트를 씌우면 **RMSE**가 됩니다.

> **결론**: RMSE는 "그럴듯해 보여서" 만든 공식이 아니라, **정규분포 가정으로부터 수학적으로 필연적으로 도출**되는 지표입니다.

### 1.5 정규분포 가정이 깨지면? (RMSE vs MAE)

2장에서 세 모델의 성능을 비교했을 때 RMSE를 사용했습니다.

| 모델 | 훈련셋 RMSE | 교차 검증 평균 RMSE |
|------|------------|-------------------|
| 선형 회귀 | 약 68,688 | 약 7만 |
| 결정트리 | 0 | 약 6만7천 |
| 랜덤 포레스트 | 약 17,474 | 약 4만7천 |

만약 오차가 정규분포를 심하게 벗어나는 경우(이상치가 많은 경우 등)에는 RMSE 대신 **MAE(평균 절대 오차)** 가 더 적합합니다.  
RMSE는 오차를 제곱하기 때문에 큰 오차 하나가 전체 지표를 크게 왜곡할 수 있기 때문입니다.

In [ ]:
import numpy as np

# 이상치가 있을 때 RMSE vs MAE 비교
errors_normal  = np.array([1, -2, 3, -1, 2])    # 정상적인 오차
errors_outlier = np.array([1, -2, 3, -1, 100])  # 이상치 포함

def rmse(errors):
    return np.sqrt(np.mean(errors**2))

def mae(errors):
    return np.mean(np.abs(errors))

print('Without Outliers')
print(f'  RMSE: {rmse(errors_normal):.2f}')
print(f'  MAE:  {mae(errors_normal):.2f}')
print()
print('With Outliers (one error = 100)')
print(f'  RMSE: {rmse(errors_outlier):.2f}  ← Very sensitive to outliers')
print(f'  MAE:  {mae(errors_outlier):.2f}   ← Relatively less sensitive to outliers')

---
## 2. 시그모이드와 베르누이 분포

### 2.1 로지스틱 회귀의 출력값은 왜 확률인가?

4장에서 배운 로지스틱 회귀는 이름에 "회귀"가 붙어 있지만 실제로는 **분류 모델**입니다.  
핵심은 출력값을 **확률**로 해석한다는 점입니다.

1장에서 다룬 스팸 메일 분류를 예로 들어 봅시다.

> "무료 당첨 클릭하세요"라는 이메일이 스팸일 확률이 **0.97**  
> "내일 점심 같이 먹을래요?"라는 이메일이 스팸일 확률이 **0.02**

날씨 예보에서 "내일 비가 올 확률 90%"와 똑같은 방식으로,
모델이 얼마나 확신하는지를 0~1 사이 숫자로 표현한 것입니다.

### 2.2 베르누이 분포와의 연결

스팸이냐 아니냐처럼 **결과가 딱 두 가지인 상황**을 통계에서 **베르누이 분포**라고 부릅니다.  
동전 던지기와 같은 구조입니다.

- 동전 던지기: 앞면(1) 또는 뒷면(0), 확률 $p$
- 스팸 분류: 스팸(1) 또는 정상(0), 확률 $p$

차이점이 있다면, 동전은 $p = 0.5$로 고정이지만,  
스팸 분류에서는 **이메일마다 $p$가 다릅니다**.

로지스틱 회귀가 하는 일은 바로 이것입니다.

> **"이 이메일의 특성들을 보고, 이 특정 이메일에 맞는 $p$를 추정하겠다."**

이메일마다 앞면이 나올 확률이 다른 동전을 던지는 것에 비유할 수 있습니다.

### 2.3 시그모이드 함수는 어디서 나오는가?

선형 회귀의 출력값은 $-\infty$에서 $+\infty$ 사이 어디든 나올 수 있습니다.  
이 값을 그대로 확률로 사용할 수는 없습니다. **확률은 반드시 0과 1 사이**여야 하니까요.

여기서 **로짓 변환**이 등장합니다. 확률 $p$를 제약 없는 실수로 변환하는 방법입니다.

$$\text{로짓}(p) = \log\left(\frac{p}{1-p}\right)$$

| $p$ | 로짓 값 |
|-----|--------|
| 0.01 | -4.6 |
| 0.5  | 0.0  |
| 0.99 | +4.6 |

로지스틱 회귀는 이 로짓값이 선형 회귀의 출력과 같다고 놓습니다.  
이걸 $p$에 대해 거꾸로 풀면, 자연스럽게 **시그모이드 함수**가 나옵니다.

$$\sigma(x) = \frac{1}{1 + e^{-x}}$$

> **결론**: 시그모이드는 누군가 그럴듯해 보여서 갖다 붙인 게 아니라,  
> **베르누이 분포 가정 + 로짓 변환**에서 수학적으로 필연적으로 도출되는 함수입니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

x = np.linspace(-8, 8, 200)
y = sigmoid(x)

plt.figure(figsize=(8, 4))
plt.plot(x, y, 'b-', linewidth=2)
plt.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
plt.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
plt.axhline(y=0.0, color='gray', linestyle='--', alpha=0.5)
plt.title('Sigmoid Function: Maps Any Input to [0, 1]')
plt.xlabel('Linear Regression Output (Internal Score)')
plt.ylabel('Probability (Spam Probability)')
plt.annotate('Strong spam features → Probability near 1', xy=(5, 0.99), xytext=(1.5, 0.83),
             arrowprops=dict(arrowstyle='->', color='red'), color='red')
plt.annotate('Weak spam features → Probability near 0', xy=(-5, 0.01), xytext=(-7.5, 0.17),
             arrowprops=dict(arrowstyle='->', color='green'), color='green')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# 시그모이드는 모델 내부 점수를 확률로 번역하는 장치
scores = [-5, -2, 0, 2, 5]
print('Internal Score → Probability (Sigmoid)')
print('-' * 40)
for s in scores:
    p = sigmoid(s)
    print(f'Score {s:+3d}  →  Prob {p:.4f}  ({"Spam" if p > 0.5 else "Normal"})')

---
## 3. 소프트맥스와 카테고리 분포

### 3.1 다중 분류 문제

3장에서 다룬 MNIST 손글씨 분류는 0~9까지 **10개의 클래스** 중 하나를 고르는 문제입니다.  
이건 스팸/정상처럼 두 가지 결과가 아니라, **여러 가지 결과 중 하나**가 나오는 상황입니다.

이런 상황을 통계에서 **카테고리 분포**라고 부릅니다.  
주사위 던지기처럼 결과가 여러 범주 중 하나로 나오고, 모든 확률의 합이 1이 되어야 합니다.

베르누이 분포와 비교하면 이렇습니다.

| | 베르누이 분포 | 카테고리 분포 |
|---|---|---|
| 비유 | 동전 던지기 | 주사위 던지기 |
| 결과 수 | 2가지 | 여러 가지 |
| 머신러닝 예 | 스팸 분류 | MNIST 손글씨 분류 |

### 3.2 소프트맥스 함수는 어디서 나오는가?

베르누이 분포에서 **로짓 변환**이 시그모이드를 낳았듯,  
카테고리 분포에서는 **로그 변환**이 소프트맥스를 낳습니다.

각 클래스의 확률 $p_k$에 로그를 취하고, 모델 내부 점수 $z_k$로부터 확률을 계산하면 자연스럽게 소프트맥스가 나옵니다.

$$p_k = \frac{\exp(z_k)}{\sum_{j} \exp(z_j)}$$

| | 베르누이 분포 | 카테고리 분포 |
|---|---|---|
| **클래스 수** | 2개 | 여러 개 |
| **변환 방법** | 로짓 변환 | 로그 변환 |
| **결과 함수** | 시그모이드 | 소프트맥스 |
| **제약 조건** | $0 \leq p \leq 1$ | 모든 $p_k$의 합 = 1 |

> 베르누이 분포가 카테고리 분포의 특수한 경우(클래스 2개)이듯,  
> 로지스틱 회귀도 소프트맥스 회귀의 특수한 경우입니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def softmax(z):
    exp_z = np.exp(z - np.max(z))  # 수치 안정성을 위해 최댓값을 뺌
    return exp_z / exp_z.sum()

# MNIST: 숫자 5 이미지에 대한 모델 내부 점수 (가상)
scores = np.array([-2.1, -3.5, -1.2, -2.8, -1.5, 4.3, -2.0, -1.8, -2.3, -1.9])
probs = softmax(scores)

print('MNIST Digit Classification Result (Image of digit 5)')
print('-' * 45)
for digit, (s, p) in enumerate(zip(scores, probs)):
    bar = '█' * int(p * 40)
    marker = ' ← predicted' if digit == np.argmax(probs) else ''
    print(f'Digit {digit}: score={s:5.1f}  prob={p:.4f}  {bar}{marker}')

print(f'\nSum of all probabilities: {probs.sum():.6f} (always 1)')

plt.figure(figsize=(8, 4))
colors = ['red' if i == np.argmax(probs) else 'steelblue' for i in range(10)]
plt.bar(range(10), probs, color=colors)
plt.title('Softmax Output: Probability for Each Digit (Sum = 1)')
plt.xlabel('Digit Class')
plt.ylabel('Probability')
plt.xticks(range(10))
plt.grid(True, alpha=0.3, axis='y')
plt.show()

---
## 4. 정리

| 알고리즘 | 확률 가정 | 도출되는 함수 | 활용 예 |
|---------|---------|-------------|--------|
| 선형 회귀 | 오차 ~ **정규분포** | **RMSE** | 캘리포니아 주택가격 예측 (2장) |
| 로지스틱 회귀 | 결과 ~ **베르누이 분포** | **시그모이드** | 스팸 분류, 이진 분류 (4장) |
| 소프트맥스 회귀 | 결과 ~ **카테고리 분포** | **소프트맥스** | MNIST 손글씨 분류 (3장) |

세 알고리즘 모두 공통된 설계 철학을 가집니다.

> **"데이터가 어떤 확률분포를 따른다고 가정하고,**  
> **그 가정 아래에서 데이터가 나올 가능성을 최대로 만드는 파라미터를 찾는다."**

RMSE, 시그모이드, 소프트맥스는 모두 이 원칙에서 수학적으로 필연적으로 도출된 결과입니다.